这次主要学习的内容是用阿拉伯语语种微调大模型，在此之前我们先来了解一下大模型一些概念

# 前言  
之前我们一直强调，大语言模型（LLM）是概率生成系统。 他是具有能力边界的 ，具体来说包含以下内容：  
* **知识时效性**：模型知识截止于训练数据时间点  
大语言模型的所有知识来自于其训练语料库，知识截⽌于模型训练数据收集的时间点。习惯上，我们以发布日期或数据集收集期限来标注。例如，某模型的训练截止日期为2023年9月，那么该模型无法知晓此后发生的事件、发表的新研究以及实时动态。  

* **推理局限性**：本质是概率预测而非逻辑运算，复杂数学推理易出错  
LLM本质上为基于概率的符号预测系统，而非传统意义上的逻辑演算引擎。它们通过最大化给定上下文的下一个词概率，实现文本续写。但对于复杂逻辑推理、精确的数学运算或长跨度依赖，LLM会出现回答错误问题  

* **专业领域盲区**：缺乏垂直领域知识  
尽管在通用文本上表现优异，但LLM对于一些垂直领域（如量子化学、特定医学子领域、专业法律条款）缺乏足够深度的知识储备。其语料库中高级专业文献占比有限，模型难以像资深领域专家那样精确回答  

* **幻觉现象**：可能生成看似合理但实际错误的内容  
幻觉（Hallucination）是指模型在缺乏事实依据的情况下，生成看似合理但实际错误或虚构的内容。常见表现包括：引用不存在的文献或报告、杜撰人物、事件或统计数据、错误的参考链接或数据源  


![Image Name](https://cdn.kesci.com/upload/sw6z4givcy.png?imageView2/0/w/960/h/960)  



# 大模型微调介绍  
- **大模型微调的概念**  
  - 在预训练模型基础上进行特定领域或任务的二次训练  
  - 利用迁移学习原理，将通用知识迁移到特定领域  
  - 保持模型的基础能力，同时获得新的领域知识  
  
	
![Image Name](https://cdn.kesci.com/upload/sts7ovu067.png?imageView2/0/w/960/h/960)  


- **微调的必要性**  

  - 提升模型在特定领域的表现  
  - 适应特定任务的需求  
  - 解决领域特定的语言理解问题  
  - 因为大模型的参数量非常大，训练成本非常高，每家公司都去从头训练一个自己的大模型，这个事情的性价比非常低  

- **传统微调的局限**  
  - 计算资源需求大：需要更新所有模型参数  
  - 训练不稳定：容易出现灾难性遗忘  
  - 存储开销大：每个任务都需要完整的模型副本  
  - 部署复杂：难以在资源受限环境中使用  
  
	![Image Name](https://cdn.kesci.com/upload/sw6z99dagk.jpg?imageView2/0/w/960/h/960)  


# 微调流程  
1. 微调数据来源  
2. 数据处理流程  
3. 模型准备  
4. 确定微调方法  
5. 确定微调参数  
6. 一边微调，一边评估性能  
![Image Name](https://cdn.kesci.com/upload/sts8e02na8.png?imageView2/0/w/960/h/960)  

在本教程中，我们选择lora作为微调方法  


# Lora核心优势  
  - **训练效率**  
    - 显存占用降低：通常只需要原始训练的1/3显存  
    - 训练速度提升：参数量减少导致计算量降低  
    - 收敛更快：较小的参数空间更容易优化  
  
  - **灵活性**  
    - 快速任务切换：只需切换不同的LoRA权重  
    - 多任务融合：可以组合多个LoRA权重  
    - 增量学习：易于添加新的任务适配器  
  
  - **实用性**  
    - 原始模型可重用：不改变基础模型参数  
    - 存储效率高：每个任务只需存储小型LoRA权重  
    - 部署友好：可动态加载不同任务的适配器  

- **技术特点**  
  - **矩阵分解策略**  
    - 选择合适的秩r：权衡性能和效率  
    - 缩放因子α：控制LoRA更新的影响程度  
    - 自适应更新：可针对不同层设置不同参数  
  
  - **模块选择**  
    - 注意力层：q_proj, k_proj, v_proj, o_proj  
    - 前馈层：可选择性应用  
    - 自定义模块：根据任务特点选择  
  
  - **训练优化**  
    - 梯度裁剪：防止训练不稳定  
    - 学习率调度：采用预热和衰减策略  
    - 正则化：防止过拟合  

# lora原理示意图  

![Image Name](https://cdn.kesci.com/upload/sts8ztl2w8.png?imageView2/0/w/960/h/960)  

输入x → W（冻结）→ 输出  
          ↓  
          BA（可训练）→ 输出修正  
最终输出 = Wx + BAx  




# lora具体算法  
![Image Name](https://cdn.kesci.com/upload/sts8cwimua.png?imageView2/0/w/960/h/960)  


## OpenDataLab阿拉伯语专业领域数据集  
   包含新闻、技术文档等多种类型文本  
   经过质量筛选和清洗的高质量数据  


![Image Name](https://cdn.kesci.com/upload/sw8agi1845.png?imageView2/0/w/960/h/960)  


### 数据处理流程  


##### 文本预处理  
```
def clean_text(text):  
    """清理文本内容"""  
    if not text:  
        return text  
    
    # 正则表达式是一种文本匹配模式,下面详细解释每一步:  
    
    # 1. 处理连续换行符  
    # re.sub()函数用于替换文本,接受3个参数:  
    # - 第1个参数 r'\n+' 表示:  
    #   \n 代表换行符  
    #   + 表示匹配1个或多个连续的换行符  
    # - 第2个参数 '\n' 表示用单个换行符替换  
    # - 第3个参数是要处理的文本  
    # strip()去除文本首尾的空格  
    text = re.sub(r'\n+', '\n', text.strip())  
    
    # 2. 处理连续空格  
    # r'\s+' 表示:  
    # \s 代表任意空白字符(空格、制表符等)  
    # + 表示匹配1个或多个连续的空白字符  
    # 用单个空格替换所有连续的空白字符  
    text = re.sub(r'\s+', ' ', text)  
    
    # 3. 移除HTML标签，正则表达式 r'<[^>]+>'  
    # 1) r'' 表示这是一个原始字符串,不会对反斜杠\进行转义处理  
    # 2) < 就是匹配HTML标签的开始符号 <  
    # 3) [^>] 是一个字符集:  
    #    - [] 表示匹配其中的任意一个字符  
    #    - ^ 在[]内表示"非",即取反  
    #    - 所以[^>]表示匹配任何不是>的字符  
    # 4) + 表示"一个或多个",即重复前面的[^>]一次或多次  
    # 5) > 就是匹配HTML标签的结束符号 >  
    #  
    # 举例说明:  
    # 原文本: "这是<p>一个段落</p>"  
    # - <p> 会被匹配,因为它符合模式:<加上任意非>字符(这里是p)再加上>  
    # - </p> 也会被匹配,因为它符合模式:<加上任意非>字符(这里是/p)再加上>  
    #  
    # re.sub()会把所有匹配到的内容替换为空字符串'',所以最后变成:  
    # "这是一个段落"  
    text = re.sub(r'<[^>]+>', '', text)  
    return text.strip()  

```

这段代码的主要功能是清理文本数据，具体包括：  
* 移除HTML标签，使文本更干净  
* 将多个连续的空白字符（空格、换行等）替换为单个空格  
* 移除所有特殊字符，只保留字母、数字、下划线和空格  
* 最后去除文本首尾的空白字符  

这种文本清理通常用于：  
* 网页内容抓取后的数据清洗  
* 文本分析前的预处理  
* 数据标准化处理  
* 自然语言处理任务的数据准备  

#### 数据转换  
##### 指令格式构建  
```
    # 转换数据  
    converted_data = []  
    for item in tqdm(data, desc="转换数据"):  
        instruction = item.get('instruction', '')  
        input_text = item.get('input', '')  
        output = item.get('output', '')  
        
        # 构建标准格式的训练样本  
        training_sample = {  
            "prompt": instruction,  
            "input": instruction+input_text,  
            "output": output  
        }  
        
        converted_data.append(training_sample)  
    
    # 保存转换后的数据  
    logger.info(f"保存转换后的数据到 {output_file}...")  
    with open(output_file, 'w', encoding='utf-8') as f:  
        json.dump(converted_data, f, ensure_ascii=False, indent=2)  
```

##### 编码操作  

![Image Name](https://cdn.kesci.com/upload/sw8b1ixdhh.png?imageView2/0/w/960/h/960)  
```

dataset = Dataset.from_dict({  
        "input_ids": tokenized_data["input_ids"],  
        "attention_mask": tokenized_data["attention_mask"],  
        "labels": labels  
    })  

for i in range(len(batch)):  
            pad_len = max_len - len(input_ids[i])  
            if pad_len > 0:  
                input_ids[i] += [tokenizer.pad_token_id] * pad_len  
                attention_mask[i] += [0] * pad_len  
                labels[i] += [-100] * pad_len  # 使用-100填充标签  
```

#### 数据集划分  
```
    split_dataset = dataset.train_test_split(test_size=val_ratio, seed=42)  
```

![Image Name](https://th.bing.com/th/id/R.b042ff1e6e387a3ba210a0c848ec202b?rik=Y8Kqoq3iYEcIvA&pid=ImgRaw&r=0)  

  比例设置  
        训练集：90%  
        验证集：10%

#### 模型准备  
   - **基础模型加载**  
     ```python  
     model = AutoModelForCausalLM.from_pretrained(  
         "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",  
         trust_remote_code=True,  
         torch_dtype=torch.float16  
     )  
     ```


   - **Tokenizer配置**  
     ```python  
     tokenizer = AutoTokenizer.from_pretrained(  
         "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",  
         trust_remote_code=True  
     )  
     tokenizer.pad_token = tokenizer.eos_token  
     ```
		 
#### 确定微调方法  
   - **LoRA参数设置**  
     ```python  
     lora_config = LoraConfig(  
         r=8,                      # LoRA秩  
         lora_alpha=32,           # 缩放因子  
         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  
         lora_dropout=0.1,  
         bias="none",  
         task_type=TaskType.CAUSAL_LM  
     )  
     ```

#### 确定微调参数  
   - **优化器设置**  
     ```python  
     optimizer = AdamW(  
         model.parameters(),  
         lr=5e-4,                # 学习率  
         weight_decay=0.01       # 权重衰减  
     )  
     ```
   
   - **学习率调度**  
     ```python  
     scheduler = get_linear_schedule_with_warmup(  
         optimizer,  
         num_warmup_steps=200,   # 预热步数  
         num_training_steps=num_training_steps  
     )  
     ```
   
   - **训练超参数**  
     - 批次大小：4-8（根据显存）  
     - 训练轮次：3-5  
     - 梯度累积：4-8步  
     - 梯度裁剪：1.0  

#### 确定评估指标  

- **困惑度(Perplexity)**  
  - **数学定义**  
   
![Image Name](https://cdn.kesci.com/upload/stxhdu1b7k.png?imageView2/0/w/960/h/960)  

    其中：  
    - \( N \) 是序列长度  
    - \( p(x_i|x_{<i}) \) 是模型对位置 i 处token的预测概率  
    - \( x_{<i} \) 表示位置 i 之前的所有token  
  
  - **代码实现**  
    ```python  
    def calculate_perplexity(model, eval_dataloader):  
        total_loss = 0  
        total_tokens = 0  
        for batch in eval_dataloader:  
            outputs = model(**batch)  
            total_loss += outputs.loss * batch["input_ids"].size(0)  
            total_tokens += batch["input_ids"].ne(tokenizer.pad_token_id).sum()  
        return torch.exp(total_loss / total_tokens)  
    ```

- **领域适应性评估**  
  - **术语覆盖率(Term Coverage Rate, TCR)**  
    - **数学定义**  

![Image Name](https://cdn.kesci.com/upload/stxhe4ia80.png?imageView2/0/w/960/h/960)  

其中：  
- \( T_r \) 是响应中出现的术语集合  
- \( T_d \) 是领域术语词典  
- \( |·| \) 表示集合的基数  
    
    - **代码实现**  
      ```python  
      def calculate_term_coverage(response, domain_terms):  
          covered_terms = sum(1 for term in domain_terms if term in response)  
          return covered_terms / len(domain_terms)  
      ```
  
  - **术语密度(Term Density, TD)**  
    - **数学定义**  
![Image Name](https://cdn.kesci.com/upload/stxheq6kfj.png?imageView2/0/w/960/h/960)  

其中：  
- \( |T_r| \) 是响应中术语的出现次数  
- \( |W_r| \) 是响应中的总词数  
    
    - **代码实现**  
      ```python  
      def calculate_term_density(response, domain_terms):  
          term_count = sum(1 for term in domain_terms if term in response)  
          return term_count / len(response.split())  
      ```
  
  - **响应质量评估**  
    - **语义相似度(Semantic Similarity, SS)**  
![Image Name](https://cdn.kesci.com/upload/stxhjtpg0n.png?imageView2/0/w/960/h/960)  

其中：  
 - \( E(·) \) 是文本的嵌入向量  
 - \( \cdot \) 表示点积  
 - \( ||·|| \) 表示向量的L2范数  

```python  
    response_embedding = self.sentence_model.encode([response])  
    prompt_embedding = self.sentence_model.encode([text])  
    similarity = cosine_similarity(response_embedding, prompt_embedding)[0][0]  
    metrics["response_quality"].append(similarity)  
```
    
#### 一边训练一边评估  
   - **梯度更新**  
     ```python  
		 for epoch in range(num_epochs):  
			 model.train()  
       total_loss = 0  
       progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")  
       print(len(train_dataloader))  
       for step, batch in enumerate(progress_bar):  
           batch = {k: v.to(device) for k, v in batch.items()}  
           outputs = model(**batch)  
           loss = outputs.loss  
           total_loss += loss.item()  
            
           loss.backward()  
           torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
           optimizer.step()  
           lr_scheduler.step()  
           optimizer.zero_grad()  
            
           progress_bar.set_postfix({"loss": loss.item()})  
            
            # 评估  
            if step % eval_steps == 0 and step > 0:  
                model.eval()  
                
                # 1. 计算验证集困惑度  
                val_perplexity = evaluator.calculate_domain_perplexity(model, val_dataloader)  
                
                # 2. 评估领域适应性  
                domain_metrics = {}  
                for lang, prompts in unlabeled_eval_prompts.items():  
                    metrics = evaluator.evaluate_domain_adaptation(model, prompts, lang)  
                    domain_metrics[lang] = metrics  
                
                # 3. 计算综合指标  
                avg_domain_score = np.mean([  
                    m["avg_term_coverage"] * 0.4 +  
                    m["avg_term_density"] * 0.3 +  
                    m["avg_response_quality"] * 0.3  
                    for m in domain_metrics.values()  
                ])  
                
                # 记录当前学习率  
                current_lr = optimizer.param_groups[0]["lr"]  
                
                metrics = {  
                    "epoch": epoch + 1,  
                    "step": step,  
                    "val_perplexity": val_perplexity.item() if isinstance(val_perplexity, torch.Tensor) else float(val_perplexity),  
                    "domain_adaptation_score": float(avg_domain_score),  
                    "learning_rate": float(current_lr),  
                    "domain_metrics": convert_metrics_to_json_serializable(domain_metrics)  
                }  
                metrics_log.append(metrics)  
                
                logger.info(f"\n验证集困惑度: {val_perplexity:.4f}")  
                logger.info(f"领域适应性得分: {avg_domain_score:.4f}")  
                logger.info(f"当前学习率: {current_lr:.6f}")  
                
                # 4. 保存最佳模型  
                combined_score = avg_domain_score/val_perplexity  
                if combined_score > best_metrics["domain_adaptation"]/best_metrics["val_perplexity"]:  
                    best_metrics["val_perplexity"] = float(val_perplexity)  
                    best_metrics["domain_adaptation"] = float(avg_domain_score)  
                    model.save_pretrained(best_model_path)  
                    logger.info(f"保存新的最佳模型！困惑度={val_perplexity:.4f}, 领域得分={avg_domain_score:.4f}")  
                
                model.train()  
     ```
   

目前为止我们介绍了普遍的微调流程、LoRA原理、LoRA流程内容，接下来我们进入用阿拉伯语语种数据去微调deepseek实战阶段